# AI weather forecasting in one hour: Storm Boris, September 2024

**CAS in Machine Learning, ETH Zurich**

Storm Boris was a Vb cyclone that stalled over Central Europe between 12 and 16 September 2024
and produced the heaviest rainfall on record in parts of Czechia, Austria and Poland. We will
forecast it with a modern AI weather model and then downscale the result to kilometre scale.

The whole session runs on a free Colab **T4 GPU**.

### What you will do

| # | Section | Time |
|---|---|---|
| 0 | Set up the runtime and install `earth2studio` | 4 min |
| 1 | Pull ERA5 reanalysis for the case from Google's ARCO archive | 7 min |
| 2 | Run a **Pangu-Weather** daily forecast on the GPU | 10 min |
| 3 | Verify it against ERA5, persistence and climatology | 8 min |
| 4 | Downscale to **2.2 km, hourly** with CorrDiff, including a small ensemble | 22 min |
| 5 | Exercises | 3 min |

### What you should take away

1. A modern global AI forecast model is a few hundred MB of weights and one `for` loop over
   autoregressive steps. The hard parts are the data plumbing and the verification, not the
   model call.
2. Forecast **skill** is meaningless without a **baseline**. Persistence and climatology are the
   two you always compute first.
3. Pangu-Weather is *deterministic*: one initial condition, one forecast, no free ensemble. Real
   uncertainty quantification in this notebook comes later, from CorrDiff's diffusion sampler --
   a useful contrast with a probabilistic model like FourCastNet 3, which gets an ensemble for
   free by re-seeding.
4. Global models run at 0.25 degrees (~28 km) and, in Pangu's case, carry no moisture or
   precipitation variables at all. That is far too coarse -- and too incomplete -- for Alpine
   precipitation. Generative downscaling closes both gaps at once: 2.2 km, hourly, with rain.

> **Runtime**: `Runtime > Change runtime type > T4 GPU`. Do this before running anything.

---
## 0. Runtime and installation

First, check what hardware we actually got.

In [ ]:
import subprocess
import sys

print("Python", sys.version.split()[0])

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu.returncode != 0:
    raise SystemExit(
        "No GPU visible. Go to Runtime > Change runtime type > T4 GPU, then re-run this cell."
    )
print("GPU:", gpu.stdout.strip())

Now the install. Two things are worth understanding here:

- **Pangu-Weather** ships as an ONNX graph, run through `onnxruntime-gpu` rather than PyTorch. No
  training framework needed -- this is a genuinely different inference stack from the diffusion
  model in section 4, and worth noticing as a contrast.
- **CorrDiff** needs `nvidia-physicsnemo` and `natten` (neighbourhood attention, the mechanism
  that makes km-scale attention affordable). physicsnemo has to come from the exact git revision
  earth2studio pins for its `cosmo` extra, not from the PyPI release of the same version number --
  more on why in section 4.

We pin the **already-installed torch** with a constraints file so the resolver cannot decide to
swap Colab's CUDA-matched build for a different one mid-install.

This takes 2-4 minutes. Start it and read on.

In [ ]:
%pip install -q uv

import os
import pathlib
import re
import subprocess
import sys
import time
import urllib.request

import torch

print(f"python {sys.version.split()[0]}   torch {torch.__version__}   cuda {torch.version.cuda}")

# Model packages are multi-GB and Colab bandwidth varies; the 300 s default
# aborts a download that is nearly complete.
os.environ["EARTH2STUDIO_PACKAGE_TIMEOUT"] = "1800"

# Pin the torch Colab already has, so nothing downgrades or reinstalls it.
pathlib.Path("constraints.txt").write_text(f"torch=={torch.__version__.split('+')[0]}\n")


def pip(*args, **env):
    """uv pip install that actually raises on failure, unlike a bare ! magic."""
    subprocess.run(
        ["uv", "pip", "install", "--system", "-q", "--constraint", "constraints.txt", *args],
        check=True,
        env={**os.environ, **env},
    )

In [ ]:
t0 = time.perf_counter()
pip(
    "earth2studio==0.17.0",
    # physicsnemo MUST come from the revision earth2studio pins for the cosmo
    # extra, not from PyPI. Released 2.2.0 prebuilds the RoPE tables in the DiT
    # and sources them from the per-call attn_kwargs, so CorrDiff's set_domain
    # rebind (which mutates attn_kwargs_forward) leaves them at the construction
    # grid and any cropped domain dies with a rope_cos/rope_sin shape mismatch.
    "nvidia-physicsnemo @ git+https://github.com/NVIDIA/physicsnemo.git"
    "@ced75d93d014f70bb691372788eee2d201171c12",
    "einops>=0.8.1", "nvtx", "cartopy",
    # ONNX runtime for Pangu. Like NATTEN below, it has to be in place before
    # earth2studio is imported: models/px/pangu.py runs `import onnxruntime`
    # at module import time and caches the failure.
    #
    # Pinned to the last CUDA-12 build: 1.29 requires nvidia-*-cu13 and looks
    # for libcublas.so.13 / libcudart.so.13, which Colab's CUDA 12.8 does not
    # have. 1.26.0 pins nvidia-*-cu12 and satisfies earth2studio's >=1.21.0.
    "onnxruntime-gpu==1.26.0", "onnx==1.21.0",
)
print(f"everything else installed in {time.perf_counter() - t0:.0f} s")

# NATTEN, needed by the CorrDiff downscaler in section 4. It has to go in before
# earth2studio is imported for the first time: earth2studio.models.dx does
# `import natten` at module import time and caches the failure, so installing it
# later leaves section 4 broken until the kernel restarts.
#
# Resolve the wheel from the index rather than pinning a version. NATTEN publishes
# only for the two most recent torch builds, and which version exists for a given
# torch/CUDA/Python triple changes often enough that a hard-coded pin goes stale.
# "%2B" in the href is the "+" of the wheel's local version tag.
_tv = torch.__version__.split("+")[0].split(".")
TORCH_TAG = f"torch{_tv[0]}{_tv[1]}0"
CUDA_TAG = "cu" + torch.version.cuda.replace(".", "")
CP = f"cp{sys.version_info.major}{sys.version_info.minor}"

index = urllib.request.urlopen("https://whl.natten.org/", timeout=60).read().decode()
matches = re.findall(
    rf'href="([^"]*natten-([\d.]+)%2B{TORCH_TAG}{CUDA_TAG}-{CP}-{CP}-linux_x86_64\.whl)"',
    index,
)
if not matches:
    raise RuntimeError(
        f"No NATTEN wheel for {TORCH_TAG}/{CUDA_TAG}/{CP}. Check https://whl.natten.org/ "
        "-- see the README for the Flex Attention fallback."
    )
NATTEN_URL, NATTEN_VERSION = max(matches, key=lambda m: [int(x) for x in m[1].split(".")])
pip(NATTEN_URL)
print(f"natten {NATTEN_VERSION} installed for {TORCH_TAG} {CUDA_TAG} {CP}")

> If the next cell raises an `ImportError` mentioning numpy, the install upgraded a package that
> was already loaded. `Runtime > Restart session`, then re-run from the cell below (you do **not**
> need to re-run the install).

In [ ]:
import gc
import warnings
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from earth2studio import run
from earth2studio.data import ARCO, WB2Climatology, fetch_data
from earth2studio.io import XarrayBackend
from earth2studio.models.px import Pangu24
from earth2studio.utils.coords import map_coords

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False

print("earth2studio ready, device =", DEVICE, "| cartopy =", HAS_CARTOPY)

---
## 1. The case: Storm Boris

A **Vb track** (pronounced "five-b", after van Bebber's 1891 classification) is a cyclone that
runs from the Gulf of Genoa north-east around the Alps. It is the single most dangerous
precipitation pattern in Central Europe, because it wraps warm, very moist Mediterranean air
onto the northern flank of the Alps and the Bohemian mountains, where the terrain forces it to
rise. The 2002 Elbe flood and the 2013 Danube flood were both Vb events.

Boris was a textbook case with an extra ingredient: a **cut-off low** detached from the
mid-latitude jet, which meant it had nothing to steer it and simply sat over the region for four
days. Rainfall totals exceeded 400 mm in parts of the Czech and Austrian mountains.

Our experiment: initialise a forecast at **00 UTC on 9 September 2024**, several days before the
rain begins, and see whether the model builds the cut-off low in the right place at the right time.

In [ ]:
# ---- Experiment configuration -------------------------------------------------
QUICK = False  # True -> shorter forecast and fewer diffusion samples, for a first test run

INIT = np.datetime64("2024-09-09T00:00:00")  # forecast initialisation
LEAD_HOURS = 144  # 6 days, taking us to the peak of the event
N_SAMPLES = 4  # CorrDiff diffusion-mode ensemble size (section 4)

if QUICK:
    LEAD_HOURS, N_SAMPLES = 72, 2

# Variables to keep from the forecast. Pangu predicts 69 of them (winds, temperature,
# geopotential -- no moisture, no precipitation); we subset to what we plot and verify.
VARS = ["msl", "t2m", "u10m", "v10m", "z500"]

# European window used for plotting and verification.
LAT_RANGE = (32.0, 62.0)
LON_RANGE = (-12.0, 32.0)

# Downscaling target: CorrDiff's finest settings, 2.2 km over the Alps.
DOWNSCALE_TIME = np.datetime64("2024-09-15T00:00:00")  # anchor hour, peak of the event
DOWNSCALE_RES = "rea2"  # "rea2" = 2.2 km, "rea6" = 6 km
DOWNSCALE_BBOX = dict(lat_min=45.5, lat_max=49.5, lon_min=8.0, lon_max=16.5)
# Six consecutive ARCO-ERA5 hours ending at DOWNSCALE_TIME -- ARCO is native hourly,
# so this is the finest temporal cadence available, not a subsample of something coarser.
DOWNSCALE_TIMES = DOWNSCALE_TIME - np.arange(5, -1, -1) * np.timedelta64(1, "h")

NSTEPS = LEAD_HOURS // 24  # Pangu-Weather (24h variant) takes 24 h steps
VALID = INIT + np.arange(NSTEPS + 1) * np.timedelta64(24, "h")

print(f"init {INIT}  ->  {VALID[-1]}   ({NSTEPS} daily steps)")

### Where the data comes from

We use **ARCO-ERA5**, Google's cloud-optimised copy of the ECMWF ERA5 reanalysis. It is a single
Zarr store on public GCS covering 1940 to the present at 0.25 degrees and **native hourly**
resolution, so we can slice out the exact fields and times we need without downloading an archive
-- including, later, the hourly sequence CorrDiff downscales.

ERA5 plays two different roles below, and it is worth keeping them apart:

- the **initial condition** the forecast starts from, and
- the **truth** we score the forecast against at every lead time.

The `Cache` wrapper below avoids re-fetching the same ERA5 slice from GCS every time we touch it
-- the same daily forecast initial condition, for instance, gets read once here and reused by the
verification step later.

In [ ]:
arco = ARCO()
climatology = WB2Climatology("1990-2019_6h_1440x721.zarr")


class Cache:
    """Memoise a data source so the same fields are downloaded once, not once per member."""

    def __init__(self, source):
        self.source = source
        self._store = {}

    def __call__(self, time, variable):
        t = np.atleast_1d(np.asarray(time, dtype="datetime64[ns]"))
        v = tuple(str(x) for x in np.atleast_1d(variable))
        key = (t.tobytes(), v)
        if key not in self._store:
            self._store[key] = self.source(t, list(v))
        return self._store[key]


era5 = Cache(arco)


def to_dataset(da, subset=True):
    """earth2studio DataArray -> tidy Dataset on -180..180 longitudes, cropped to Europe."""
    ds = da.to_dataset("variable")
    ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby("lon")
    if subset:
        ds = ds.sel(lat=slice(LAT_RANGE[1], LAT_RANGE[0]), lon=slice(*LON_RANGE))
    return ds

In [ ]:
def make_axes(nrows, ncols, figsize):
    """Grid of map axes, with coastlines if cartopy is available."""
    kw = {"subplot_kw": {"projection": ccrs.PlateCarree()}} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, **kw)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        if HAS_CARTOPY:
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.6)
            ax.set_extent([*LON_RANGE, *LAT_RANGE], crs=ccrs.PlateCarree())
    return fig, axes


def draw_wind(ax, ds, stride=6, scale=420, **kw):
    """10 m wind as arrows, subsampled so the field stays readable."""
    sub = ds.isel(lat=slice(None, None, stride), lon=slice(None, None, stride))
    ax.quiver(
        sub.lon, sub.lat, sub.u10m, sub.v10m,
        color="0.15", scale=scale, width=0.003, **GEO, **kw,
    )


# Passed to every pcolormesh/contour/quiver call so the data is georeferenced when cartopy is on.
GEO = {"transform": ccrs.PlateCarree()} if HAS_CARTOPY else {}

Now pull three snapshots through the life cycle of the storm. Total column water vapour (`tcwv`)
shows the moisture plume; mean sea level pressure (`msl`) shows the cyclone wrapping up.

This is a live download from GCS and takes a minute or two.

In [ ]:
SNAPSHOTS = np.array(
    ["2024-09-11T12", "2024-09-13T12", "2024-09-15T00"], dtype="datetime64[ns]"
)

overview = to_dataset(era5(SNAPSHOTS, ["msl", "tcwv"]))
overview

In [ ]:
fig, axes = make_axes(1, 3, (16, 4.2))

for ax, t in zip(axes, SNAPSHOTS):
    snap = overview.sel(time=t)
    m = ax.pcolormesh(
        snap.lon, snap.lat, snap.tcwv, vmin=5, vmax=45, cmap="BrBG", shading="auto", **GEO
    )
    cs = ax.contour(
        snap.lon, snap.lat, snap.msl / 100, levels=np.arange(960, 1044, 4),
        colors="k", linewidths=0.6, **GEO,
    )
    ax.clabel(cs, cs.levels[::2], fontsize=6, fmt="%d")
    ax.set_title(str(t)[:13].replace("T", " ") + " UTC", fontsize=10)

fig.colorbar(m, ax=axes, shrink=0.8, label="total column water vapour [kg m$^{-2}$]")
fig.suptitle("ERA5: Storm Boris, moisture plume and sea level pressure", y=1.02)
plt.show()

Read the sequence left to right. On 11 September the moisture is still a plume streaming north out
of the Mediterranean. By the 13th the low has closed off over Central Europe and is rotating that
plume around itself. By the 15th it has barely moved, which is exactly the problem: the same air
keeps being lifted over the same mountains for days.

---
## 2. Forecasting with Pangu-Weather

**Pangu-Weather** (Bi et al., 2023) was one of the first AI models to beat operational numerical
weather prediction on headline scores, and it is built very differently from the diffusion model
we will use in section 4:

- It is a **3D Earth-Specific Transformer**: attention over a 3D grid of (pressure level,
  latitude, longitude) tokens, with an explicit height axis rather than treating pressure levels
  as extra channels.
- It is **deterministic**. One initial condition in, one forecast out -- no internal noise to
  re-seed. If you want an ensemble from a model like this, you have to manufacture spread
  yourself, by perturbing the initial condition or the weights (exactly what the
  `ai-models-ensembles` pipeline this notebook borrows from does for a living).
- It ships as an **ONNX graph** and runs through `onnxruntime-gpu`, not native PyTorch. The
  checkpoint is a single 1.1 GB `.onnx` file.
- The variant we use here, `Pangu24`, takes a **24-hour** autoregressive step -- one forward pass
  per day, rather than 6-hourly. Companion `Pangu6`/`Pangu3` checkpoints exist for finer steps,
  cascaded together, but at roughly double the VRAM for the pair; `Pangu24` alone is what fits
  comfortably on a free T4.

One real limitation: Pangu's 69 variables are winds, temperature and geopotential on 13 pressure
levels plus a handful of surface fields. No humidity, no precipitation. Keep that in mind -- it is
exactly the gap section 4 exists to fill.

In [ ]:
t0 = time.perf_counter()
model = Pangu24.load_model(Pangu24.load_default_package()).to(DEVICE)
print(f"loaded in {time.perf_counter() - t0:.0f} s")

if DEVICE == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {(total - free) / 1e9:.1f} GB used / {total / 1e9:.1f} GB total")

### Running the rollout

Same idea as any autoregressive forecast: feed the model a state, get the state a step later, feed
that back in. `earth2studio.run.deterministic` is that loop plus the data plumbing --
`output_coords` below subsets what gets written out (all 69 variables at 721x1440 for 6 steps
would still be small here, but the habit of subsetting is worth keeping).

This is a single deterministic forecast, six 24-hour steps, no re-seeding: on a T4 it took about
8 seconds per step in testing, so under a minute of GPU compute for the whole 6-day forecast --
most of the wall-clock time is the checkpoint download and the ARCO fetch.

In [ ]:
# Pangu's native latitude grid, cut to our European window. Longitude stays global
# here (the window straddles the prime meridian, so it is not contiguous on a
# 0..360 grid) and gets cropped after the run.
lat_full = np.linspace(90.0, -90.0, 721)
lat_keep = lat_full[(lat_full >= LAT_RANGE[0]) & (lat_full <= LAT_RANGE[1])]

out_coords = OrderedDict({"variable": np.array(VARS), "lat": lat_keep})

io = XarrayBackend()
t0 = time.perf_counter()
run.deterministic(
    [INIT], NSTEPS, model, era5, io, output_coords=out_coords, device=DEVICE, verbose=False
)
print(f"{NSTEPS} daily steps: {time.perf_counter() - t0:.0f} s")

In [ ]:
fcst = io.root.isel(time=0)
fcst = (
    fcst.assign_coords(lon=(((fcst.lon + 180) % 360) - 180))
    .sortby("lon")
    .sel(lon=slice(*LON_RANGE))
)

print(f"{fcst.nbytes / 1e6:.0f} MB in memory")
fcst

### Watching the storm build, day by day

Six panels, one per forecast day: `t2m` shaded, `msl` contoured, 10 m wind as arrows. This is the
model's entire view of the event -- everything it predicted, laid out in sequence.

In [ ]:
fig, axes = make_axes(2, 3, (16, 8.5))

for ax, t in zip(axes, VALID[1:]):
    day = fcst.sel(lead_time=t - INIT)
    m = ax.pcolormesh(
        day.lon, day.lat, day.t2m, vmin=270, vmax=300, cmap="RdYlBu_r", shading="auto", **GEO
    )
    cs = ax.contour(
        day.lon, day.lat, day.msl / 100, levels=np.arange(960, 1044, 4),
        colors="k", linewidths=0.6, **GEO,
    )
    ax.clabel(cs, cs.levels[::2], fontsize=6, fmt="%d")
    draw_wind(ax, day)
    ax.set_title(f"+{int((t - INIT) / np.timedelta64(1, 'h'))} h  ({str(t)[:10]})", fontsize=10)

fig.colorbar(m, ax=axes, shrink=0.7, label="2 m temperature [K]")
fig.suptitle(f"Pangu-Weather forecast initialised {str(INIT)[:13].replace('T', ' ')} UTC", y=1.0)
plt.show()

Watch the pressure contours (`msl`) for the cut-off low closing off over Central Europe rather
than continuing to track east with the jet, and the wind arrows for the field wrapping into a
closed circulation around it -- that closed, stationary loop is the Vb pattern's signature, and
it is what pins the rain over one place for days. Position and timing errors of a day or so, or a
few hundred kilometres, are normal at this range for a single deterministic run; day 6 in a
blocked pattern is genuinely hard.

### Did it get the storm?

One direct check: the day-6 forecast against what actually happened, side by side.

In [ ]:
T_CHECK = VALID[-1]
truth_check = to_dataset(era5(np.array([T_CHECK]), ["msl", "t2m", "u10m", "v10m"])).isel(time=0)
fc_check = fcst.isel(lead_time=-1)  # same instant as T_CHECK

fig, axes = make_axes(1, 2, (11, 4.6))
vmin, vmax = 270, 300

for ax, (title, ds) in zip(axes, [("ERA5 truth", truth_check), (f"Pangu24, +{LEAD_HOURS} h", fc_check)]):
    m = ax.pcolormesh(ds.lon, ds.lat, ds.t2m, vmin=vmin, vmax=vmax, cmap="RdYlBu_r", shading="auto", **GEO)
    cs = ax.contour(
        ds.lon, ds.lat, ds.msl / 100, levels=np.arange(960, 1044, 4),
        colors="k", linewidths=0.6, **GEO,
    )
    ax.clabel(cs, cs.levels[::2], fontsize=6, fmt="%d")
    draw_wind(ax, ds)
    ax.set_title(title, fontsize=10)

fig.colorbar(m, ax=axes, shrink=0.8, label="2 m temperature [K]")
fig.suptitle(f"Valid {str(T_CHECK)[:13].replace('T', ' ')} UTC", y=1.03)
plt.show()

---
## 3. Verification: is it any good?

A forecast map that looks plausible tells you almost nothing. The only question that matters is
whether the forecast beats the cheap alternatives, so we always score against two baselines:

- **Persistence**: assume nothing changes from the initial condition. This is very hard to beat at
  short lead times and trivial to beat at long ones.
- **Climatology**: the 1990-2019 average for this day of year and hour. This is impossible to beat
  at long lead times, because that is where every forecast eventually converges. When your RMSE
  reaches the climatological RMSE, your forecast has no information left.

A useful forecast lives between those two curves. Where it crosses the climatology line is,
roughly, the limit of its predictability for this case.

Pangu's native step is 24 hours, so we verify at each of the six daily steps -- a coarser curve
than a 6-hourly model would give, but that coarseness is the model's actual temporal resolution,
not a choice we are making to save downloads.

In [ ]:
vtimes = VALID
lead_h = fcst.lead_time / np.timedelta64(1, "h")
fc_v = fcst


def align_to(ds, ref):
    """Put ds on ref's exact lat/lon/lead_time labels, failing loudly on a shape mismatch."""
    assert (ds.sizes["lat"], ds.sizes["lon"]) == (ref.sizes["lat"], ref.sizes["lon"]), (
        f"grid mismatch: {dict(ds.sizes)} vs {dict(ref.sizes)}"
    )
    return ds.rename(time="lead_time").assign_coords(
        lat=ref.lat, lon=ref.lon, lead_time=ref.lead_time
    )


truth_v = align_to(to_dataset(era5(vtimes, VARS)), fc_v)
clim_v = align_to(to_dataset(climatology(vtimes, VARS)), fc_v)

print("verification times:", len(vtimes))

In [ ]:
# Latitude weighting: a grid cell at 60N covers half the area of one at the equator.
W = np.cos(np.deg2rad(fc_v.lat))


def wrmse(a, b):
    return np.sqrt(((a - b) ** 2).weighted(W).mean(("lat", "lon")))


def wacc(f, o, c):
    """Anomaly correlation coefficient of forecast f against truth o, relative to climatology c."""
    fa, oa = f - c, o - c
    num = (fa * oa).weighted(W).mean(("lat", "lon"))
    den = np.sqrt(
        (fa**2).weighted(W).mean(("lat", "lon")) * (oa**2).weighted(W).mean(("lat", "lon"))
    )
    return num / den


persistence = truth_v.isel(lead_time=0).drop_vars("lead_time")

scores = {
    "Pangu24": wrmse(fc_v, truth_v),
    "persistence": wrmse(persistence, truth_v),
    "climatology": wrmse(clim_v, truth_v),
}

In [ ]:
SHOW = [("z500", 1 / 9.81, "500 hPa geopotential height [m]"),
        ("msl", 1 / 100, "mean sea level pressure [hPa]"),
        ("t2m", 1.0, "2 m temperature [K]")]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
styles = {"Pangu24": dict(color="C0", lw=2.0, marker="o", ms=4),
          "persistence": dict(color="0.45", lw=1.4, marker="o", ms=3),
          "climatology": dict(color="C3", lw=1.4, ls=":", marker="o", ms=3)}

for ax, (var, scale, label) in zip(axes, SHOW):
    for name, sc in scores.items():
        ax.plot(lead_h, sc[var] * scale, label=name, **styles[name])
    ax.set_xlabel("lead time [h]")
    ax.set_title(label, fontsize=10)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("RMSE")
axes[0].legend(fontsize=8)
fig.suptitle(f"Latitude-weighted RMSE over Europe, initialised {str(INIT)[:13]} UTC", y=1.02)
plt.tight_layout()
plt.show()

Read these curves as a story about information. Six points is a coarse sample of the underlying
curve -- there is no data on what happened in between two daily steps -- but the shape should
still be legible.

At **day 1** persistence is competitive: the atmosphere has not had time to change much, so
"nothing happens" is a decent forecast. Pangu should already be ahead, but not dramatically.

Through the **middle days** the gap should open up -- this is where a forecast earns its keep.

At **long lead times** the forecast curve bends towards climatology. When your RMSE reaches the
climatological RMSE, the forecast has no information left at that lead time; watching where that
happens for each variable tells you how far out this particular event was predictable.

Now the anomaly correlation, the score operational centres actually quote. ACC measures whether
you predicted the right *departure* from normal. The conventional threshold for a useful synoptic
forecast is **ACC = 0.6**.

In [ ]:
acc = wacc(fc_v, truth_v, clim_v)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(lead_h, acc.z500, "C0", lw=2.0, marker="o", ms=4, label="Pangu24")
ax.axhline(0.6, color="C3", ls=":", label="useful-forecast threshold")
ax.set_xlabel("lead time [h]")
ax.set_ylabel("ACC")
ax.set_title("500 hPa geopotential anomaly correlation", fontsize=10)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

**A caveat you should insist on.** One storm, one initialisation, one deterministic run. RMSE and
ACC computed from six points are illustrative, not statistically robust -- a real evaluation
scores tens to hundreds of initialisation dates. Rankings built on a single case are how bad
papers get written.

We are not computing a spread-skill ratio here, because Pangu gives us nothing to take a spread
of -- it is one forecast, not an ensemble. That diagnostic returns in section 4, where CorrDiff's
diffusion sampler genuinely does produce multiple physically plausible outcomes, and where the
question "is my ensemble the right size" actually has an answer to check.

---
## 4. Downscaling to 2.2 km, hourly, with CorrDiff

Here is the uncomfortable truth about everything above: **Pangu-Weather never predicted any rain,
and could not have.** Its 69 variables are winds, temperature and geopotential -- no humidity, no
precipitation, nothing about clouds. And a 0.25 degree grid cell is about 28 km wide, whereas the
Alpine valleys and ridges that decide where 400 mm of rain lands are 5 to 20 km apart. The global
model gives you the synoptic setup; it cannot give you the impact.

**CorrDiff** closes both gaps at once. It is a *corrector diffusion* model with two stages:

1. a **regression** network predicting the conditional mean of the high-resolution field given the
   coarse one, which recovers everything deterministically predictable from the terrain, and
2. a **diffusion** network sampling the residual, the small-scale structure that is physically
   plausible but not uniquely determined by the coarse input.

The variant we use, `CorrDiffCosmoEra5`, maps ERA5 onto **COSMO-REA**, the DWD regional reanalysis
over Europe, at either 6 km (`rea6`) or 2.2 km (`rea2`). Crucially it outputs `TOT_PRECIP` --
finally, the field this whole event is actually about.

We will run it at its finest settings in both dimensions: **2.2 km** spatially (`rea2`) and, since
ARCO-ERA5 is native hourly, **hourly** rather than the once-a-day cadence Pangu gave us. Then we
switch on the diffusion half and look at what a small ensemble of downscaled precipitation fields
actually looks like.

First, free the GPU. Pangu is still holding several GB -- and since `onnxruntime`'s memory arena
sits outside PyTorch's own allocator, `torch.cuda.empty_cache()` alone will not touch it; dropping
the reference first is what actually releases it.

In [ ]:
model = None
gc.collect()
torch.cuda.empty_cache()

if DEVICE == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free: {free / 1e9:.1f} GB / {total / 1e9:.1f} GB")

CorrDiff's backbone is a diffusion transformer that uses **neighbourhood attention** (each token
attends only to its local window, which is what makes attention affordable at 2 km resolution).
That needs the `natten` library, which ships compiled CUDA kernels rather than a pure-Python
implementation, so the wheel has to match Colab's exact torch, CUDA and Python ABI. It was already
installed back in section 0, and it had to be: `earth2studio.models.dx` runs `import natten` when
it is first imported and caches the result, so installing it here instead would leave this section
broken until the kernel restarted.

Now load the model and restrict it to our domain.

`set_domain` is the feature that makes this usable on a T4. The network is a DiT with rotary
position embeddings, so it is **crop-size agnostic**: it was trained at a fixed 2.2 km resolution
but will run on any sub-block of that grid without retraining. Asking for the Alps and Bohemia
instead of the full European domain cuts the memory by a large factor.

We load two copies of the network below, one per `mode`: `"mean"` for the fast single-pass
regression -- what we use for the resolution comparison and the hourly sequence -- and
`"diffusion"` for the generative ensemble later. Both come from the same downloaded package, so
the second load is cheap.

In [ ]:
from earth2studio.models.dx import CorrDiffCosmoEra5

t0 = time.perf_counter()
cd_package = CorrDiffCosmoEra5.load_default_package()
downscaler = CorrDiffCosmoEra5.load_model(
    cd_package, device=DEVICE, mode="mean", resolution=DOWNSCALE_RES
)
alps = downscaler.set_domain(**DOWNSCALE_BBOX).to(DEVICE)
print(f"loaded in {time.perf_counter() - t0:.0f} s")

ic = alps.input_coords()
oc = alps.output_coords(ic)
print(f"ERA5 input : {len(ic['lat'])} x {len(ic['lon'])} cells, {len(ic['variable'])} variables")
print(f"COSMO output: {np.asarray(oc['lat']).shape} cells at ~2.2 km")

Note the asymmetry in those two numbers. A few thousand coarse input cells become a few hundred
thousand output cells. The model is not interpolating: it is generating structure that is
consistent with the terrain and with what COSMO-REA looks like in this synoptic situation.

Feed it the ERA5 analysis at the peak of the event, and compare against the 0.25 degree input.

In [ ]:
x, coords = fetch_data(
    source=arco,
    time=np.array([DOWNSCALE_TIME], dtype="datetime64[ns]"),
    variable=ic["variable"],
    device=DEVICE,
)

# fetch_data returns (time, lead_time, variable, lat, lon). This diagnostic takes one
# validity time per frame, so drop the singleton lead_time axis before handing it over.
x = x[:, 0]
coords = OrderedDict((k, v) for k, v in coords.items() if k != "lead_time")
x, coords = map_coords(x, coords, ic)

t0 = time.perf_counter()
with torch.inference_mode():
    hires, hires_coords = alps(x, coords)
print(f"downscaled in {time.perf_counter() - t0:.1f} s -> {tuple(hires.shape)}")

# (batch, sample, time, variable, H, W) -> (variable, H, W)
field = hires[0, 0, 0].float().cpu().numpy()
out_names = [str(v) for v in hires_coords["variable"]]
lat2d = np.asarray(hires_coords["lat"])
lon2d = np.asarray(hires_coords["lon"])
print("available outputs:", out_names)

In [ ]:
def pick(names, *candidates):
    for c in candidates:
        if c in names:
            return names.index(c)
    raise KeyError(f"none of {candidates} in {names}")


era5_names = [str(v) for v in ic["variable"]]
era5_t2m = x[0, era5_names.index("t2m")].float().cpu().numpy()
cosmo_t2m = field[pick(out_names, "t2m")]

kw = {"subplot_kw": {"projection": ccrs.PlateCarree()}} if HAS_CARTOPY else {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), **kw)
extent = [DOWNSCALE_BBOX["lon_min"], DOWNSCALE_BBOX["lon_max"],
          DOWNSCALE_BBOX["lat_min"], DOWNSCALE_BBOX["lat_max"]]

tmin, tmax = np.percentile(cosmo_t2m, [1, 99])

p0 = axes[0].pcolormesh(ic["lon"], ic["lat"], era5_t2m, vmin=tmin, vmax=tmax,
                        cmap="RdYlBu_r", shading="auto", **GEO)
axes[0].set_title("ERA5 2 m temperature, 0.25 deg (~28 km)", fontsize=10)
fig.colorbar(p0, ax=axes[0], shrink=0.75, label="K")

p1 = axes[1].pcolormesh(lon2d, lat2d, cosmo_t2m, vmin=tmin, vmax=tmax,
                        cmap="RdYlBu_r", shading="auto", **GEO)
axes[1].set_title(f"CorrDiff -> COSMO-{DOWNSCALE_RES.upper()} 2 m temperature (~2.2 km)", fontsize=10)
fig.colorbar(p1, ax=axes[1], shrink=0.75, label="K")

for ax in axes:
    if HAS_CARTOPY:
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="0.3")
        ax.set_extent(extent, crs=ccrs.PlateCarree())

fig.suptitle(f"Valid {str(DOWNSCALE_TIME)[:13].replace('T', ' ')} UTC", y=1.03)
plt.tight_layout()
plt.show()

The right panel is the payoff. The Alps stop being a smooth bump and become a mountain range:
valley floors are warm, ridges are cold, and the temperature field carries the imprint of terrain
that simply does not exist in the 0.25 degree input. The model learned that relationship from four
years of COSMO-REA and applies it to a state it has never seen.

### Precipitation, hour by hour

Now the field Pangu could not give us at all, and at ARCO's native hourly cadence rather than a
single snapshot. One forward call, batched over six consecutive hours ending at the peak time.

A note on units before we plot: `TOT_PRECIP` comes back in the package's native metres (via the
`CosmoLexicon` scale factor), which we convert to millimetres below. What exact accumulation
window it represents (the preceding hour, an instantaneous rate, something else) is a COSMO-REA
convention documented on the DWD pages linked in `CorrDiffCosmoEra5`'s own docstring -- worth
checking before you use this for anything beyond a classroom demo.

In [ ]:
xh, coordsh = fetch_data(
    source=arco, time=np.array(DOWNSCALE_TIMES, dtype="datetime64[ns]"),
    variable=ic["variable"], device=DEVICE,
)
xh = xh[:, 0]  # drop the singleton lead_time axis; the size-6 time axis stays
coordsh = OrderedDict((k, v) for k, v in coordsh.items() if k != "lead_time")
xh, coordsh = map_coords(xh, coordsh, ic)

t0 = time.perf_counter()
with torch.inference_mode():
    hires_h, hcoords_h = alps(xh, coordsh)
print(f"downscaled {len(DOWNSCALE_TIMES)} hours in {time.perf_counter() - t0:.1f} s -> {tuple(hires_h.shape)}")

names_h = [str(v) for v in hcoords_h["variable"]]
tp_h = hires_h[0, 0, :, pick(names_h, "tp")].float().cpu().numpy() * 1e3  # m -> mm

In [ ]:
fig, axes = make_axes(2, 3, (15, 8.5))
vmax = max(1.0, float(np.percentile(tp_h, 99)))

for ax, t, tp in zip(axes, DOWNSCALE_TIMES, tp_h):
    m = ax.pcolormesh(
        lon2d, lat2d, np.ma.masked_less(tp, 0.05), vmin=0, vmax=vmax, cmap="GnBu", shading="auto", **GEO
    )
    if HAS_CARTOPY:
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="0.3")
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_title(str(t)[:13].replace("T", " ") + " UTC", fontsize=10)

fig.colorbar(m, ax=axes, shrink=0.7, label="TOT_PRECIP [mm]")
fig.suptitle(f"CorrDiff precipitation at ~2.2 km, hourly, {DOWNSCALE_RES.upper()}", y=1.0)
plt.show()

This is the field a hydrologist would actually want: precipitation concentrated on windward
slopes, evolving hour by hour, at a resolution where individual valleys are resolved. Neither of
those two things -- rain at all, or this much spatial detail -- existed anywhere upstream of this
cell.

### How sure is the model? Switching on the diffusion sampler

Everything above used `mode="mean"`: the regression network's single best guess, fast but smooth
by construction -- it is a conditional *mean*, which structurally under-represents extremes and
carries no notion of its own uncertainty.

`mode="diffusion"` is the other half of CorrDiff: an EDM/Heun sampler that draws
`number_of_samples` independent realisations from the learned conditional distribution
`p(y | x)`, each seeded separately. Unlike Pangu, this genuinely is an ensemble -- multiple
different, physically plausible high-resolution outcomes from the same coarse input -- and it is
far more expensive: 18 sampler steps per sample, so `N_SAMPLES` full network evaluations, run at a
single hour rather than the full sequence above to keep this inside the session's time budget.

In [ ]:
t0 = time.perf_counter()
downscaler_diff = CorrDiffCosmoEra5.load_model(
    cd_package, device=DEVICE, mode="diffusion", resolution=DOWNSCALE_RES
)
alps_diff = downscaler_diff.set_domain(**DOWNSCALE_BBOX).to(DEVICE)
alps_diff.number_of_samples = N_SAMPLES
print(f"loaded in {time.perf_counter() - t0:.0f} s, {N_SAMPLES} samples")

xd, coordsd = fetch_data(
    source=arco, time=np.array([DOWNSCALE_TIME], dtype="datetime64[ns]"),
    variable=ic["variable"], device=DEVICE,
)
xd = xd[:, 0]
coordsd = OrderedDict((k, v) for k, v in coordsd.items() if k != "lead_time")
xd, coordsd = map_coords(xd, coordsd, ic)

t0 = time.perf_counter()
with torch.inference_mode():
    hires_d, hcoords_d = alps_diff(xd, coordsd)
print(f"{N_SAMPLES} samples in {time.perf_counter() - t0:.0f} s -> {tuple(hires_d.shape)}")

names_d = [str(v) for v in hcoords_d["variable"]]
# (batch, sample, time, variable, H, W) -> (sample, H, W)
tp_samples = hires_d[0, :, 0, pick(names_d, "tp")].float().cpu().numpy() * 1e3

In [ ]:
fig, axes = make_axes(1, N_SAMPLES + 1, (4.3 * (N_SAMPLES + 1), 4.4))
vmax = max(1.0, float(np.percentile(tp_samples, 99)))

# make_axes() defaults every panel to the Europe-wide extent; narrow each one to
# the Alps bbox we actually downscaled, same as the plots earlier in this section.
for ax in axes:
    if HAS_CARTOPY:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

for k in range(N_SAMPLES):
    m = axes[k].pcolormesh(
        lon2d, lat2d, np.ma.masked_less(tp_samples[k], 0.05),
        vmin=0, vmax=vmax, cmap="GnBu", shading="auto", **GEO,
    )
    axes[k].set_title(f"sample {k}", fontsize=10)

spread = tp_samples.std(axis=0)
ms = axes[-1].pcolormesh(lon2d, lat2d, spread, cmap="magma", shading="auto", **GEO)
axes[-1].set_title("spread (std across samples)", fontsize=10)
fig.colorbar(ms, ax=axes[-1], shrink=0.75, label="mm")

fig.colorbar(m, ax=axes[:-1], shrink=0.75, label="TOT_PRECIP [mm]")
fig.suptitle(
    f"CorrDiff diffusion ensemble, {DOWNSCALE_RES.upper()}, "
    f"{str(DOWNSCALE_TIME)[:13].replace('T', ' ')} UTC", y=1.02,
)
plt.show()

Look for two things. **Agreement on the large-scale pattern**: all four samples should put the
heaviest rain over roughly the same slopes, because that is the part the coarse ERA5 input
actually determines. **Disagreement on the fine texture**: exactly which ridge gets the highest
single value, the precise shape of individual cells -- that is the part that is genuinely
underdetermined by a 28 km input, and the diffusion sampler is showing you its own uncertainty
about it rather than papering over it with a single smooth guess. The spread map on the right is
where that disagreement concentrates -- if it is close to zero everywhere, the ensemble collapsed
and is not saying anything a single regression sample would not.

### Two honest caveats

**We downscaled the ERA5 analysis, not our own forecast.** The natural pipeline is Pangu ->
CorrDiff, and earth2studio supports exactly that through `run.diagnostic`. It does not work out of
the box here: CorrDiff needs 47 ERA5 inputs including surface pressure `sp` and the 100 m wind
components, and Pangu's 69 outputs include none of those three. The fix is to insert
`earth2studio.models.dx.DerivedSurfacePressure` (for `sp`; the 100 m winds have no clean
substitute and would need to be dropped from the input list, degrading the downscaler slightly).
That variable bookkeeping is exercise 4 below.

**A sharper field is not automatically a better one.** The regression mode is smoother than
reality and systematically under-predicts extremes. The diffusion mode produces realistic-looking
fields with the right amount of small-scale variance, but any individual sample is one draw from a
distribution, not the truth. Verifying downscaled precipitation properly needs neighbourhood or
distribution-based scores such as FSS or CRPS, never a point-to-point RMSE, which would rank the
blurry regression field higher for the wrong reason.

---
## 5. Exercises

1. **Move the initialisation.** Set `INIT` to `2024-09-11T00` and re-run section 2. Does that
   change where the ACC curve crosses 0.6? With only six daily points the curve is coarse -- does
   that change your confidence in reading off an exact crossing?

2. **Manufacture a Pangu ensemble.** Pangu is deterministic, so getting a spread out of it takes
   real work: use `earth2studio.run.ensemble` with an initial-condition perturbation
   (`earth2studio.perturbation.SphericalGaussian`) to build an N-member ensemble, then compute its
   spread-skill ratio the way section 3 discussed for CorrDiff. Does IC-perturbation spread look
   as physically structured as the diffusion ensemble's spread map in section 4?

3. **Grow the diffusion ensemble.** Set `N_SAMPLES` to 8 or 16 and re-run the diffusion cell. Does
   the spread map change shape, or just get statistically smoother? At `rea6` (6 km) instead of
   `rea2`, how much further can you push the sample count before a T4 struggles?

4. **Chain the models.** Build the Pangu -> `DerivedSurfacePressure` -> CorrDiff pipeline described
   above and downscale the day-6 *forecast* rather than the ERA5 analysis. Compare it against what
   section 4 already produced, to separate downscaling error from forecast error.

5. **Change the metric.** Compute a fractions skill score for the downscaled precipitation instead
   of RMSE, and show that the ranking of regression mode against diffusion mode depends on which
   score you choose.

### Where to go next

- earth2studio documentation and examples: <https://nvidia.github.io/earth2studio>
- Pangu-Weather: <https://doi.org/10.1038/s41586-023-06185-3>
- CorrDiff: <https://arxiv.org/abs/2309.15214>
- WeatherBench 2, the standard benchmark and its precomputed forecasts for 2018-2022:
  <https://sites.research.google/weatherbench/>
- ARCO-ERA5: <https://github.com/google-research/arco-era5>

---
## Instructor preflight

One-off hardware check. Delete this section before giving the notebook to students.

In [ ]:
# INSTRUCTOR PREFLIGHT -- delete before the class
"""Probe 3: can Pangu24 and CorrDiff diffusion coexist on one T4?

Pangu24 FITS (measured: 15.30/15.6 GB peak, 8 s/step, 24 h step, 1.10 GB ONNX).
FengWu OOMed in the ORT arena; FCN3 and Atlas ruled out earlier.

Two open questions, both of which decide the session design:
  A. ONNX Runtime allocates outside torch's allocator, so torch.cuda.empty_cache()
     cannot free it. If deleting the Pangu session does not return the ~15 GB,
     CorrDiff cannot load in the same kernel and the notebook needs a restart
     between sections -- which is fine, but I have to write it that way.
  B. What diffusion mode costs: 18 EDM/Heun steps x N samples on the Alps domain.

Run after cells 0-5 in a FRESH kernel.
"""

import gc
import time
import traceback
from collections import OrderedDict

import numpy as np
import torch

from earth2studio import run
from earth2studio.data import ARCO, fetch_data
from earth2studio.io import XarrayBackend
from earth2studio.utils.coords import map_coords

REPORT = []
BBOX = dict(lat_min=45.5, lat_max=49.5, lon_min=8.0, lon_max=16.5)
N_SAMPLES = 4


def log(k, v):
    REPORT.append(f"{k:<30} {v}")
    print(f"{k:<30} {v}")


def vram(tag):
    free, total = torch.cuda.mem_get_info()
    log(tag, f"{(total - free) / 1e9:.2f} / {total / 1e9:.1f} GB in use")
    return (total - free) / 1e9


# ---- A. does ORT hand the memory back? ---------------------------------------
vram("baseline VRAM")
try:
    from earth2studio.models.px import Pangu24

    model = Pangu24.load_model(Pangu24.load_default_package())
    lat = np.linspace(90.0, -90.0, 721)
    oc = OrderedDict({
        "variable": np.array(["msl", "t2m", "u10m", "v10m"]),
        "lat": lat[(lat >= 32.0) & (lat <= 62.0)],
    })
    t0 = time.perf_counter()
    run.deterministic([np.datetime64("2024-09-09T00:00:00")], 2, model, ARCO(),
                      XarrayBackend(), output_coords=oc, device="cuda", verbose=False)
    log("Pangu24 2 steps", f"{time.perf_counter() - t0:.0f} s")
    vram("VRAM with Pangu resident")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    after = vram("VRAM after deleting Pangu")
    log("ORT memory released", "YES" if after < 2.0 else "NO -- needs kernel restart")
except Exception as e:
    log("Pangu24 stage", f"FAILED {type(e).__name__}: {str(e)[:140]}")
    traceback.print_exc()


# ---- B. CorrDiff diffusion cost ----------------------------------------------
for mode in ["mean", "diffusion"]:
    try:
        from earth2studio.models.dx import CorrDiffCosmoEra5

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        t0 = time.perf_counter()
        dx = CorrDiffCosmoEra5.load_model(
            CorrDiffCosmoEra5.load_default_package(), device="cuda",
            mode=mode, resolution="rea2",
        )
        alps = dx.set_domain(**BBOX).to("cuda")
        if mode == "diffusion":
            alps.number_of_samples = N_SAMPLES
        log(f"{mode}: load", f"{time.perf_counter() - t0:.0f} s")

        ic = alps.input_coords()
        log(f"{mode}: input grid", f"{len(ic['lat'])} x {len(ic['lon'])}")
        log(f"{mode}: output grid", f"{np.asarray(alps.output_coords(ic)['lat']).shape}")

        x, coords = fetch_data(
            source=ARCO(), time=np.array(["2024-09-15T00:00:00"], dtype="datetime64[ns]"),
            variable=ic["variable"], device="cuda",
        )
        x = x[:, 0]
        coords = OrderedDict((k, v) for k, v in coords.items() if k != "lead_time")
        x, coords = map_coords(x, coords, ic)

        t0 = time.perf_counter()
        with torch.inference_mode():
            hires, hcoords = alps(x, coords)
        log(f"{mode}: forward", f"{time.perf_counter() - t0:.1f} s -> {tuple(hires.shape)}")
        log(f"{mode}: peak torch alloc", f"{torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

        names = [str(v) for v in hcoords["variable"]]
        if mode == "mean":
            log("output variables", ",".join(names))
        f = hires[0].float().cpu().numpy()  # (sample, time, variable, H, W)
        if "tp" in names:
            tp = f[:, 0, names.index("tp")] * 1e3
            log(f"{mode}: precip max [mm]", f"{tp.max():.2f}")
            if mode == "diffusion":
                log("spread across samples", f"{tp.max(axis=0).mean() - tp.min(axis=0).mean():.3f} mm mean range")
        del dx, alps, hires
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        log(f"{mode}: VERDICT", f"FAILED {type(e).__name__}: {str(e)[:140]}")
        traceback.print_exc()

print("\n\n" + "=" * 60)
print("\n".join(REPORT))